# Gemma-4 GRPO post-SFT (ARCHITECTURE.md §2.5, §3.3, §5)

Runs the repo's RL stage on a free-tier T4/P100 session. Requirements:

1. Runtime → **GPU (T4 or P100)** — no GPU exits early.
2. The repo arrives as the attached **gemma4-coding-agent-repo** dataset (no secret needed). A `GH_TOKEN` secret works as fallback.
3. Kaggle account must have **Gemma 4 accepted** on the model page.
4. **SFT first**: upload `sft_adapter.zip` (output of `kaggle/sft.ipynb`) as a Dataset input — GRPO initializes from `checkpoints/sft`.
5. Rollouts clone task repos and build their venvs on demand (`ensure_task_environment`); first update is slower.
6. 12 h/session limit (§5): adapter + optimizer save every 25 updates; re-run with `--resume auto` to continue.

Smoke first: change cell 4 to `bash scripts/run_rl.sh --resume auto --limit-updates 2`.

In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile

assert os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_GPU_RUNNING_TYPE"), (
    "Turn on the GPU runtime first (T4/P100) — ARCHITECTURE §5"
)

repo_dir = pathlib.Path("gemma4-coding-agent")

def _extract(root: pathlib.Path) -> bool:
    zips = sorted(root.rglob("gemma4-coding-agent.zip"))
    if zips:
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(".")
        if repo_dir.is_dir():
            return True
    nested = root / "gemma4-coding-agent"
    if nested.is_dir() and (nested / "pyproject.toml").exists():
        if not repo_dir.exists():
            shutil.copytree(nested, repo_dir)
        return True
    if (root / "pyproject.toml").exists() and (root / "ARCHITECTURE.md").exists():
        if not repo_dir.exists():
            shutil.copytree(root, repo_dir, ignore=shutil.ignore_patterns("*.zip", "*.tar"))
        return True
    return False

inputs = sorted(pathlib.Path("/kaggle/input").glob("*gemma4*"))
ok = any(_extract(p) for p in inputs)
if not ok and os.environ.get("GH_TOKEN"):
    subprocess.run(
        f"git clone https://{os.environ['GH_TOKEN']}@github.com/Ashura-asura/gemma4-coding-agent.git",
        shell=True, check=True,
    )
    ok = repo_dir.is_dir()
if not ok:
    import kagglehub  # notebook carries KAGGLE_* creds; dataset fetch needs no mount
    dl = pathlib.Path(kagglehub.dataset_download("ashuraasura/gemma4-coding-agent-repo"))
    ok = _extract(dl)
assert ok, f"repo bootstrap failed; /kaggle/input = {sorted(p.name for p in pathlib.Path('/kaggle/input').iterdir())}"
os.chdir(repo_dir.resolve())
# guard: CRLF in .sh breaks bash (set: pipefail: invalid option name) on stale mounts
for _sh in pathlib.Path("scripts").glob("*.sh"):
    _sh.write_bytes(_sh.read_bytes().replace(b"\r\n", b"\n"))
subprocess.run(f'"{sys.executable}" -m pip install -q -e ".[train,dev]"', shell=True, check=True)
print("repo ready at", pathlib.Path.cwd())

In [ ]:
# pre-flight (CPU): reward/rollout/GRPO tests, template render, pool/config sanity
!python -m pytest train/rl/tests eval/tests -q
!python -m train.rl.grpo --config configs/rl_config.yaml --dry-run

In [ ]:
# SFT adapter input: unzip the uploaded sft_adapter.zip over checkpoints/sft
import glob, zipfile

zips = glob.glob("/kaggle/input/**/sft_adapter.zip", recursive=True)
assert zips, "Upload sft_adapter.zip as a Dataset input (SFT notebook output)"
with zipfile.ZipFile(zips[0]) as zf:
    print("zip contains:", zf.namelist()[:3])
    zf.extractall(".")  # archive stores checkpoints/sft/...
print("adapter at", sorted(pathlib.Path("checkpoints/sft").glob("*"))[:5])

In [ ]:
# GRPO — base model resolves to kagglehub google/gemma-4/transformers/gemma-4-12b-it on Kaggle
!bash scripts/run_rl.sh --resume auto

In [ ]:
import subprocess

!du -sh checkpoints/rl
subprocess.run(
    f'"{sys.executable}" -m zipfile -c /kaggle/working/rl_adapter.zip checkpoints/rl',
    shell=True, check=True,
)
print("download rl_adapter.zip from /kaggle/working after the run")